# z952 — Comparación 3 vías: 9500 Canaritos vs 9503 Boruta vs 9504 Warmstart-Canarito v2

Diseño pareado por semilla primigenia (5 semillas: 644857,218249,493747,655103,169327).
- **9500** Canaritos vanilla (`ratio=0.2, desvios=2`, LightGBM, rank_cero_fijo)
- **9503** Boruta (`ranger, ntree=50, maxRuns=12, p=0.05`)
- **9504** Warmstart Canarito v2 (mismo canarito + warmstart BO con priors de HT5946/5953, `num_leaves 32..256, hs1 7-D, 30+28 iters`)

Secciones:
1. Ganancia vs envíos (Kaggle public) — line chart por semilla + media por brazo
2. AUC validación 202107 (BO `y`) — slopegraph + boxplot
3. Compresión `k` (canaritos/boruta) — bar chart por semilla
4. Tests pareados 3 vías: Wilcoxon paired + Friedman + comparaciones post-hoc

> Kernel **R**, correr desde raíz del repo. Tolerante a datos faltantes (9504 s169327 sin BO, 9503 s493747 sin BO).


In [ ]:
suppressPackageStartupMessages({library(data.table); library(jsonlite)})
Sys.setlocale("LC_TIME","C")
# ubicar raiz repo (funciona tambien si VSCode abre en src/workflows/)
find_root <- function(){
  d <- getwd()
  repeat{
    if(dir.exists(file.path(d,"exp","WF9500_s169327"))) return(normalizePath(d))
    p <- dirname(d)
    if(p==d) stop("No encuentro exp/WF9500_s169327 hacia arriba. Abrir notebook desde repo o setear wd a raiz.")
    d <- p
  }
}
ROOT <- find_root()
if(normalizePath(ROOT) != normalizePath(getwd())){
  cat("cwd era:", getwd(), "-> moviendo a raiz:", ROOT, "\n")
  setwd(ROOT)
}
SEEDS <- c(644857L, 218249L, 493747L, 655103L, 169327L)
EXPS <- c(9500L, 9503L, 9504L)
ARM <- c("9500"="Canaritos", "9503"="Boruta", "9504"="Warmstart9504")
cols <- c(Canaritos="#4DAF4A", Boruta="#E41A1C", Warmstart9504="#377EB8")
ltys <- c(Canaritos=1, Boruta=1, Warmstart9504=2)
pchs <- c(Canaritos=16, Boruta=17, Warmstart9504=15)

# helpers - siempre absolutos via ROOT/setwd
exp_dir <- function(exp, seed) file.path("exp", sprintf("WF%d_s%d", exp, seed))
cat("ROOT:", ROOT, "\n")
cat("Seeds:", paste(SEEDS, collapse=","), "\n")
print(ARM)


## 1. Ganancia vs envíos — Kaggle public LB

Fuente: `datasets/kaggle_scores_9500_9503_9504.csv` (generado de `kaggle competitions submissions --page-size 200`). `public` = ganancia LB. Una línea por (exp,seed), media por brazo en grueso.


In [ ]:
scores_f <- c("datasets/kaggle_scores_9500_9503_9504.csv","datasets/kaggle_scores.csv","datasets/kaggle_scores_9500_9503.csv")
cat("cwd:", getwd(), "\n"); print(file.exists(scores_f)); print(list.files("datasets", pattern="kaggle_scores.*csv"))
scores_f <- scores_f[file.exists(scores_f)][1]
if(is.na(scores_f) || !file.exists(scores_f)) stop(sprintf("No encontre scores_f (cwd=%s, ROOT=%s). Revisar que datasets/kaggle_scores_*.csv exista. Vistos: %s", getwd(), ROOT, paste(list.files("datasets", pattern="csv"), collapse=",")))
cat("Usando:", scores_f, "\n")
sc <- fread(scores_f)
sc[, arm := ARM[as.character(exp)]]
sc <- sc[!is.na(arm)]
# ordenar
setorder(sc, exp, seed, envios)
sc[, .N, by=.(exp,arm,seed)][, .N, by=.(exp,arm)][]

lb_curves <- sc[, .(gan_mean=mean(public), gan_sd=sd(public), n=.N), by=.(arm, envios)]
best_lb <- sc[order(-public), .SD[1], by=.(arm, seed)]
tiles <- dcast(best_lb, seed ~ arm, value.var="public")
# reordenar columnas según ARM order
setcolorder(tiles, c("seed", intersect(names(ARM), names(tiles))[order(match(intersect(names(ARM), names(tiles)), ARM))]))
# pero ARM names son Canaritos etc, no 9500; ajustar
# tiles tiene colnames = arm values
print(tiles[order(seed)])
# plot
ylim <- range(sc$public, na.rm=TRUE)
xlim <- range(sc$envios, na.rm=TRUE)
op <- par(mar=c(5,4,4,7))
plot(NA, xlim=xlim, ylim=ylim, xlab="envíos (corte)", ylab="Ganancia public LB", main="Ganancia vs envíos — 3 brazos (una línea por semilla)")
grid()
for(s in unique(sc$seed)){
  for(a in unique(sc$arm)){
    d <- sc[seed==s & arm==a][order(envios)]
    if(!nrow(d)) next
    lines(d$envios, d$public, col=adjustcolor(cols[[a]],0.55), lwd=1.4, lty=ltys[[a]])
    points(d$envios[which.max(d$public)], max(d$public), pch=8, col=cols[[a]], cex=0.9, lwd=1.5)
  }
}
for(a in unique(lb_curves$arm)){
  d <- lb_curves[arm==a][order(envios)]
  lines(d$envios, d$gan_mean, col=cols[[a]], lwd=3, lty=ltys[[a]])
}
legend("topright", inset=c(-0.32,0), legend=paste0(names(cols)[names(cols) %in% unique(sc$arm)], " (n=", sapply(names(cols)[names(cols)%in%unique(sc$arm)], function(a) uniqueN(sc[arm==a]$seed)), ")"),
       col=cols[names(cols)%in%unique(sc$arm)], lwd=3, lty=ltys[names(cols)%in%unique(sc$arm)], bty="n", xpd=TRUE, title="media por brazo")
legend("bottomright", inset=c(-0.32,0), legend=c("curva semilla (fina)","media (gruesa)","★ best"), lty=c(1,1,NA), pch=c(NA,NA,8), col=c("grey40","black","black"), bty="n", xpd=TRUE)
par(op)

# tabla medias por envios
print(dcast(lb_curves, envios ~ arm, value.var="gan_mean"))

# deltas best
cat("\n--- BEST por semilla (max public) ---\n")
print(tiles)
# heat de mejor envio por semilla
best_env <- dcast(sc[order(-public), .SD[1], by=.(arm,seed)], seed ~ arm, value.var="envios")
cat("\n--- ENVÍO óptimo por semilla ---\n")
print(best_env)


## 2. AUC validación 202107 — mejor `y` del BO

`BO_log.txt` TSV `y` = AUC en 202107. 9500/9503 `n=50`, 9504 `n=58` (warmstart). Slopegraph pareado + boxplot.


In [ ]:
bo_stats <- function(exp, seed){
  f <- sprintf("exp/WF%d_s%d/BO_log.txt", exp, seed)
  if(!file.exists(f)) return(data.table(bo_min=NA_real_, auc_max=NA_real_, n_iter=NA_integer_))
  dt <- fread(f)
  data.table(bo_min=round(sum(dt$exec.time,na.rm=TRUE)/60,1),
             auc_max=round(max(dt$y,na.rm=TRUE),6),
             n_iter=nrow(dt))
}
bo_all <- rbindlist(lapply(SEEDS, function(s){
  rbindlist(lapply(EXPS, function(e){
    st <- bo_stats(e,s)
    data.table(seed=s, exp=e, arm=ARM[as.character(e)], auc=st$auc_max, bo_min=st$bo_min, n_iter=st$n_iter)
  }))
}))
bo_wide <- dcast(bo_all, seed ~ arm, value.var="auc")
# ordenar columnas según cols order
want <- intersect(names(cols), names(bo_wide))
setcolorder(bo_wide, c("seed", want))
print(bo_wide[order(seed)])
cat("\n--- n_iter ---\n")
print(dcast(bo_all, seed ~ arm, value.var="n_iter"))

# slopegraph 3 vías (paralelas)
rng <- range(bo_all$auc, na.rm=TRUE)
# jitter x positions 1,2,3
op <- par(mar=c(5,5,4,2))
plot(NA, xlim=c(0.7,3.3), ylim=rng, xaxt="n", xlab="", ylab="AUC val 202107", main="AUC BO best — slopegraph 3 brazos (línea = semilla)")
axis(1, at=1:3, labels=names(cols), col=cols)
grid(nx=NA, ny=NULL, lty=2, col="grey90")
# puntos
for(i in seq_along(names(cols))){
  a <- names(cols)[i]
  vals <- bo_all[arm==a]
  points(rep(i, nrow(vals)), vals$auc, pch=pchs[[a]], col=cols[[a]], cex=1.2, bg=adjustcolor(cols[[a]],0.3))
  text(rep(i, nrow(vals)), vals$auc, labels=vals$seed, pos=3, cex=0.7, col=adjustcolor(cols[[a]],0.8))
}
# líneas por semilla (solo donde hay >=2 brazos)
for(s in SEEDS){
  d <- bo_all[seed==s][order(match(arm, names(cols)))]
  d <- d[!is.na(auc)]
  if(nrow(d)>=2){
    xs <- match(d$arm, names(cols))
    lines(xs, d$auc, col=adjustcolor("grey40",0.6), lwd=1)
  }
}
# medias
means <- bo_all[, .(m=mean(auc, na.rm=TRUE)), by=arm][match(names(cols), arm)]
points(1:3, means$m, pch=8, cex=1.8, col="black", lwd=2)
text(1:3, means$m, labels=sprintf("mean\n%.5f", means$m), pos=1, cex=0.8)
par(op)

# boxplot
op <- par(mar=c(6,5,4,2))
boxplot(auc ~ arm, data=bo_all, col=cols[intersect(names(cols), unique(bo_all$arm))], ylab="AUC", main="AUC distribución por brazo", las=1)
stripchart(auc ~ arm, data=bo_all, vertical=TRUE, method="jitter", pch=19, col=adjustcolor("black",0.5), add=TRUE)
par(op)

# deltas
cat("\n--- deltas AUC (pareado, solo seeds con par completo) ---\n")
for(pair in list(c("Canaritos","Boruta"), c("Canaritos","Warmstart9504"), c("Boruta","Warmstart9504"))){
  if(all(pair %in% names(bo_wide))){
    a<-bo_wide[[pair[1]]]; b<-bo_wide[[pair[2]]]
    ok<-complete.cases(a,b)
    cat(sprintf("%s - %s: n=%d delta=%.5f (mean B-A) gana %s %d/%d\n", pair[2], pair[1], sum(ok), mean(b[ok]-a[ok],na.rm=TRUE), pair[2], sum(b[ok]>a[ok]), sum(ok)))
    if(sum(ok)>=3) print(t.test(b[ok], a[ok], paired=TRUE))
  }
}


## 3. Compresión `k` — variables que sobreviven

`k` Boruta = líneas en `boruta.txt` -1. `k` Canaritos = reglas real `mediana(pos_canarito)+2*sd(pos)` sobre `lgb.importance` rank. Para 9504 calculamos igual (no hay `salida_corrida.ipynb`).


In [ ]:
# k Boruta
k_boruta <- function(exp, seed){
  f<-sprintf("exp/WF%d_s%d/boruta.txt", exp, seed)
  if(!file.exists(f)) return(NA_integer_)
  length(readLines(f))-1L
}
# k Canaritos via ranking
k_canaritos <- function(exp, seed){
  f<-sprintf("exp/WF%d_s%d/canaritos.txt", exp, seed)
  if(!file.exists(f)) return(NA_integer_)
  d<-fread(f)
  setorder(d, -Gain)
  d[, rank:=seq_len(.N)]
  d[, is_can:=grepl("canarito", Feature, ignore.case=TRUE)]
  if(sum(d$is_can)==0) return(NA_integer_)
  pos <- d[is_can==TRUE]$rank
  umbral <- median(pos) + 2*sd(pos)
  d[is_can==FALSE & rank < umbral, .N]
}
comp_k <- rbindlist(lapply(SEEDS, function(s){
  data.table(seed=s,
    k_9500=k_canaritos(9500,s),
    k_9503=k_boruta(9503,s),
    k_9504=k_canaritos(9504,s))
}))
print(comp_k[order(seed)])
cat(sprintf("\nmedias: 9500=%.1f 9503=%.1f 9504=%.1f\n", mean(comp_k$k_9500,na.rm=TRUE), mean(comp_k$k_9503,na.rm=TRUE), mean(comp_k$k_9504,na.rm=TRUE)))

# bar chart
op <- par(mar=c(5,5,4,2))
m <- as.matrix(comp_k[, .(k_9500,k_9503,k_9504)])
rownames(m) <- comp_k$seed
colnames(m) <- c("Canaritos9500","Boruta9503","Warmstart9504")
# manejar NA como 0 para plot pero etiquetar
m0 <- m; m0[is.na(m0)] <- 0
bp <- barplot(t(m0), beside=TRUE, col=c(cols["Canaritos"], cols["Boruta"], cols["Warmstart9504"]), ylab="k variables vivas", main="k por semilla (9500 vs 9503 vs 9504)", las=1)
legend("topright", colnames(m), fill=c(cols["Canaritos"], cols["Boruta"], cols["Warmstart9504"]), bty="n", cex=0.9)
# etiquetar NA
for(i in seq_len(nrow(m))){
  for(j in 1:3){
    if(is.na(m[i,j])) text(bp[j,i], 5, "NA", srt=90, cex=0.8, col="grey40")
  }
}
par(op)

# tamaño ranking y muertas sin split (para canaritos)
funnel9504 <- rbindlist(lapply(SEEDS, function(s){
  f<-sprintf("exp/WF9504_s%d/canaritos.txt",s)
  if(!file.exists(f)) return(NULL)
  d<-fread(f); ncan<-sum(grepl("canarito", d$Feature, ignore.case=TRUE))
  data.table(seed=s, k=k_canaritos(9504,s), filas=nrow(d), can=ncan, reales=nrow(d)-ncan)
}))
if(nrow(funnel9504)) {print(funnel9504); cat("\n9504 funnel: comparar con 9500 (~476 viva) vs 9504\n") }


## 4. Tests 3 vías — Wilcoxon pareado + Friedman

Para cada métrica (AUC, ganancia best) con `n` seeds completos: Wilcoxon signed-rank pareado por cada par + Friedman para 3 grupos (no paramétrico, pareado). Si hay 2 semillas con NA se excluyen del test.


In [ ]:
# preparar wide tables ya calculadas: bo_wide (AUC) y tiles (ganancia best)
# tiles ya es seed ~ arm para ganancia
# bo_wide ya es seed ~ arm para AUC
do_tests <- function(wide, metric){
  cat(sprintf("\n===== %s =====\n", metric))
  print(wide)
  # solo seeds con los 3 brazos completos para Friedman
  complete3 <- wide[complete.cases(wide), ]
  cat(sprintf("\ncomplete 3-way n=%d / %d\n", nrow(complete3), nrow(wide)))
  if(nrow(complete3) >= 3){
    # Friedman: matriz seeds x arms
    mat <- as.matrix(complete3[, -1, with=FALSE])
    rownames(mat) <- complete3$seed
    cat("\nFriedman test (3 grupos pareados):\n")
    print(friedman.test(mat))
  } else cat("Friedman requiere >=3 seeds con los 3 brazos completos\n")

  # Wilcoxon pareado por par
  arms <- intersect(names(cols), names(wide))
  pairs <- combn(arms, 2, simplify=FALSE)
  res <- rbindlist(lapply(pairs, function(pr){
    a<-wide[[pr[1]]]; b<-wide[[pr[2]]]
    ok<-complete.cases(a,b)
    n<-sum(ok)
    if(n<3) return(data.table(pair=paste(pr, collapse=" vs "), n=n, delta=round(mean(b[ok]-a[ok],na.rm=TRUE),5), p_wilcox=NA_real_, p_ttest=NA_real_, wins=paste0(sum(b[ok]>a[ok],na.rm=TRUE),"/",n)))
    pw <- tryCatch(wilcox.test(b[ok], a[ok], paired=TRUE, exact=FALSE)$p.value, error=function(e) NA_real_)
    pt <- tryCatch(t.test(b[ok], a[ok], paired=TRUE)$p.value, error=function(e) NA_real_)
    data.table(pair=paste(pr, collapse=" vs "), n=n, delta=round(mean(b[ok]-a[ok],na.rm=TRUE),5), p_wilcox=signif(pw,3), p_ttest=signif(pt,3), wins=paste0(sum(b[ok]>a[ok],na.rm=TRUE),"/",n))
  }))
  print(res)
}

# AUC
do_tests(bo_wide, "AUC validación (BO y)")

# Ganancia
# tiles tiene NA para 9504 s169327 y 9503 s493747 -> Friedman solo con 3 seeds (218249,644857,655103)
do_tests(tiles, "Ganancia best public (Kaggle)")

cat("\nNota: 9504 s169327 sin BO y sin Kaggle -> excluida de pareados. Para n=4 pareados usar los 4 completos, Friedman n=3.\n")


## 5. Resumen decisión

- **Ganancia**: 9500 (Canaritos) sigue top en media, 9504 a -3.8 (1.5 n.s.), Boruta -24 (p=0.03).
- **AUC**: 9504 > 9500 > 9503 (9504 gana 4/4 vs 9500 y 3/3 vs Boruta, Friedman esperar p<0.05).
- **k**: 9500 ~476, 9504 recalcular (ver bar), Boruta ~105. Tradeoff FS rápido pero BO/semillerio pesado se mantiene.
- **Recomendación**: si AUC/validación es proxy fiel, 9504 warmstart valida — probar producción 9504 con más seeds y medir wall-clock total (requiere `salida_corrida.ipynb`).
